# Tracer Context Management

The `context.py` module manages context-local tracer and run-collector callback handlers.

It provides context managers for enabling LangSmith tracing and collecting completed runs, along with a registration mechanism used by callback-manager configuration to discover context-based handlers.

## Context Variables

1. `tracing_callback_var`: Provides backward-compatible access to the former tracing callback variable.

   This variable is retained for partial compatibility but is not used by the module.

   * **Definition:**
     ```python
     tracing_callback_var: Any = None
     ```

2. `tracing_v2_callback_var`: Stores the `LangChainTracer` active in the current execution context.

   Its default value is `None`. Values are isolated through Python's `ContextVar` mechanism.

   * **Definition:**
     ```python
     tracing_v2_callback_var: ContextVar[
         LangChainTracer | None
     ] = ContextVar(
         "tracing_callback_v2",
         default=None
     )
     ```

3. `run_collector_var`: Stores the `RunCollectorCallbackHandler` active in the current execution context.

   Its default value is `None`. The variable is registered as a non-inheritable callback configuration hook when this module is imported.

   * **Definition:**
     ```python
     run_collector_var: ContextVar[
         RunCollectorCallbackHandler | None
     ] = ContextVar(
         "run_collector",
         default=None
     )
     ```

### Functions

1. `tracing_v2_enabled`: Creates a `LangChainTracer` and temporarily activates it in the current context.

   Every LangChain run created inside the context can be sent to LangSmith through the activated tracer. The yielded tracer may also be used to inspect information associated with the traced root run.

   When `example_id` is supplied as a string, it is converted into a `UUID`. The previous value of `tracing_v2_callback_var` is restored when the context exits, including when execution raises an exception.

   * **Syntax:**
     ```python
     @contextmanager
     tracing_v2_enabled(
         project_name: str | None = None, # LangSmith project name
         *,
         example_id: str | UUID | None = None, # Optional linked example identifier
         tags: list[str] | None = None, # Tags added to traced runs
         client: LangSmithClient | None = None # Optional LangSmith client
     ) -> Generator[
         LangChainTracer,
         None,
         None
     ]
     ```

2. `collect_runs`: Creates and temporarily activates a `RunCollectorCallbackHandler`.

   Runs created inside the context are collected by the yielded handler. The previous value of `run_collector_var` is restored when the context exits, including when execution raises an exception.

   * **Syntax:**
     ```python
     @contextmanager
     collect_runs() -> Generator[
         RunCollectorCallbackHandler,
         None,
         None
     ]
     ```

3. `register_configure_hook`: Registers a context variable as a callback-manager configuration hook.

   Each registered hook records the context variable, whether its handler should be inherited by child callback managers, an optional callback-handler class, and an optional environment-variable name.

   When `env_var` is supplied, `handle_class` must also be supplied so the configuration system knows which handler type to construct. A `ValueError` is raised when an environment variable is provided without a handler class.

   * **Syntax:**
     ```python
     register_configure_hook(
         context_var: ContextVar[
             Any | None
         ], # Context variable containing an optional callback handler
         inheritable: bool, # Whether child callback managers inherit the handler
         handle_class: type[
             BaseCallbackHandler
         ] | None = None, # Handler class associated with the hook
         env_var: str | None = None # Environment variable enabling the handler
     ) -> None
     ```

## Module Registration

When the module is imported, `run_collector_var` is registered through `register_configure_hook` with inheritance disabled.

This allows callback-manager configuration to include the active run collector for the current context without automatically propagating it as an inheritable handler.